In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler
import torch

load_dotenv()

# --- Adatbázis Kapcsolat ---
engine = create_engine(f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}")

query = "SELECT valid_time, temp_c, rh, wind_speed, wind_dir, sp, tp FROM era5_adatok_fix ORDER BY valid_time"
df = pd.read_sql(query, engine, parse_dates=["valid_time"])

# --- Időbélyeg kódolása (Ciklikus jellemzők) ---
# Ez segít a TimesNet-nek és TCN-nek a napi/éves ciklusok felismerésében
def encode_cyclic_time(df, col):
    # Óra kódolása (0-23, tehát 24 egység!)
    # Megjegyzés: itt érdemesebb 24-gyel osztani, mert 0-tól 23-ig 24 állapot van
    #LINEÁRIS IDŐ: Unix timestamp (másodpercek)
    # Ez őrzi meg az évek közötti trendet
    df = df.sort_values(col).reset_index(drop=True)
    df['time_linear'] = df[col].astype('int64') // 10**9
    
    hour = df[col].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Év napja (1-365 vagy 1-366)
    day_of_year = df[col].dt.dayofyear
    
    # Megnézzük, szökőév-e az adott sor dátuma
    # Ha szökőév, 366-tal osztunk, ha nem, 365-tel
    days_in_year = np.where(df[col].dt.is_leap_year, 366, 365)
    
    df['year_sin'] = np.sin(2 * np.pi * (day_of_year - 1) / days_in_year)
    df['year_cos'] = np.cos(2 * np.pi * (day_of_year - 1) / days_in_year)
    
    return df

df = encode_cyclic_time(df, "valid_time")

# --- Feature-ök meghatározása ---
# Most 10 feature-ünk van (6 eredeti + 5 ciklikus idő)
# A 'temp_c' lesz a target, de benne hagyjuk a bemenetek között is!
features = ["temp_c", "rh", "wind_speed", "wind_dir", "sp", "tp", "hour_sin", "hour_cos", "year_sin", "year_cos", "time_linear"]
target_col = ["temp_c", "rh", "wind_speed"]

# --- Skálázás ---
scaler = MinMaxScaler(feature_range=(0, 1))
df[features] = scaler.fit_transform(df[features])

# --- Adattisztítás ---
df = df.ffill().bfill() # Hiányzó értékek kitöltése az adatbázis hibák ellen

# Konvertálás float32-re a PyTorch-hoz
data_array = df[features].values.astype(np.float32)

print(f"Adatok betöltve: {data_array.shape[0]} sor, {data_array.shape[1]} jellemző.")


Adatok betöltve: 754800 sor, 11 jellemző.


In [2]:
from torch.utils.data import Dataset, DataLoader

# Eszköz kiválasztása: ha van GPU, azt használjuk, egyébként a CPU-t
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"A kód jelenleg ezt használja: {device}")
if device.type == 'cuda':
    print(f"GPU típusa: {torch.cuda.get_device_name(0)}")

class TimeSeriesDataset(Dataset):
    def __init__(self, data, input_window, output_horizon, target_idx, mask_prob=0.3, always_keep=[0, 1]):
        """
        data: A már skálázott numpy tömb (features + target)
        target_idx: Melyik oszlop a cél (pl. temp_c indexe)
        mask_prob: Milyen eséllyel maszkoljunk el paramétereket a tanítás során
        always_keep: Azok az indexek, amiket SOHA nem maszkolunk (pl. idő és temp_c)
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.input_window = input_window
        self.output_horizon = output_horizon
        self.target_idx = target_idx
        self.mask_prob = mask_prob
        self.always_keep = always_keep

    def __len__(self):
        return len(self.data) - self.input_window - self.output_horizon

    def __getitem__(self, idx):
        # Bemeneti ablak kivágása
        x = self.data[idx : idx + self.input_window].clone()
        
        # Cél ablak kivágása (a jövőbeli értékek)
        y = self.data[idx + self.input_window : idx + self.input_window + self.output_horizon, self.target_idx]

        # --- RANDOM MASKING (A 7 -> 2) ---
        # Csak tanítás közben érdemes (ha a modell.train() módban van, de itt egyszerűségért fix)
        if self.mask_prob > 0:
            for feat_idx in range(x.shape[1]):
                if feat_idx not in self.always_keep:
                    if np.random.rand() < self.mask_prob:
                        x[:, feat_idx] = -1.0 # Maszkoló érték
        
        return x, y


#------Indexek amik targetek és amik ne legyenek hiányosra generálva közben.------
#keep_indices = [features.index("temp_c"), features.index("hour_sin"), features.index("hour_cos"), features.index("year_sin"), features.index("year_cos"), features.index("time_linear"), features.index("rh"), features.index("wind_speed"), features.index("tp")]
# 1. Target index automatikusan
target_index = [features.index(col) for col in target_col]

# 2. Extra feature-ök amik nem lesznek hiányosra generálva pluszba azokon felül amik célok.
extra_cols = [
    "hour_sin",
    "hour_cos",
    "year_sin",
    "year_cos",
    "time_linear"
]

# 3. Keep indices összeállítása (Amik nem lesznek hiányosak.)
keep_indices = target_index + [features.index(col) for col in extra_cols]



#Történelmi adatok amikkel indítunk "órában"
input_window=168
#Pedikciós elvárás "órában"
output_horizon=48

# Datasetek létrehozása
full_dataset = TimeSeriesDataset(
    data_array, 
    input_window, 
    output_horizon, 
    target_idx=target_index,
    mask_prob=0.0, # 30% eséllyel tűnnek el paraméterek
    always_keep=keep_indices
)

# Split (10% tesztadat elég nagy mennyiség figyelembe véve az adathalmazt, 8év,lesz teszt amibe 1 év teszt lesz csak)
train_size = int(len(full_dataset) * 0.98)

#test_size = len(full_dataset) - train_size
#train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_dataset = torch.utils.data.Subset(full_dataset, range(0, train_size))
test_dataset = torch.utils.data.Subset(full_dataset, range(train_size, len(full_dataset)))

# DataLoader - Ez küldi az adatokat a GPU-nak batch-ekben
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Tanító batch-ek száma: {len(train_loader)}")

A kód jelenleg ezt használja: cuda
GPU típusa: Tesla T4
Tanító batch-ek száma: 11555


In [3]:
import torch.nn as nn
import torch.nn.utils.parametrizations as parametrizations

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super(ResidualBlock, self).__init__()
        padding = (kernel_size - 1) * dilation
        
        # A régi utils.weight_norm(nn.Conv1d(...)) HELYETT:
        self.conv1 = parametrizations.weight_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                               padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        
        self.conv2 = parametrizations.weight_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                               padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1,
                                 self.conv2, self.chomp2, self.relu2)
        
        # Ha a bemeneti és kimeneti csatornaszám eltér, kell egy skip-connection illesztés
        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TCNModel(nn.Module):
    def __init__(self, num_inputs, output_horizon, num_targets,
                 num_channels=[64, 64, 128], kernel_size=3):
        super(TCNModel, self).__init__()

        self.num_targets = num_targets
        self.output_horizon = output_horizon

        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers.append(ResidualBlock(in_channels, out_channels, kernel_size, dilation=dilation_size))

        self.tcn = nn.Sequential(*layers)

        # Kimeneti réteg több targetre
        self.linear = nn.Linear(num_channels[-1], output_horizon * num_targets)

    def forward(self, x):
        # x: [Batch, Time, Features]
        x = x.transpose(1, 2)  # -> [Batch, Features, Time]

        y = self.tcn(x)        # -> [Batch, Channels, Time]

        out = self.linear(y[:, :, -1])  # -> [Batch, output_horizon * num_targets]

        # Visszaalakítás 3D formára
        out = out.view(-1, self.output_horizon, self.num_targets)
        return out

# Modell inicializálása
num_targets = len(target_index) 
model_tcn = TCNModel( num_inputs=len(features), output_horizon=output_horizon, num_targets=num_targets ).to(device)


In [4]:
import torch.optim as optim
import time

# 1. Beállítások
criterion = nn.MSELoss() # A hiba mérése
optimizer = optim.Adam(model_tcn.parameters(), lr=0.001) # Az algoritmus, ami állítja a súlyokat
epochs = 10 # Kezdésnek 10 kör elég lesz

print(f"Tanítás megkezdése a(z) {device} eszközön...")

for epoch in range(epochs):
    start_time = time.time()
    model_tcn.train() # Modell tanítási módba állítása
    running_loss = 0.0
    
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        # Adatok átküldése a GPU-ra
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # Gradiens törlése (minden lépésnél tiszta lappal indulunk)
        optimizer.zero_grad()
        
        # Előrejelzés (Forward pass)
        outputs = model_tcn(batch_x)
        
        # Hiba kiszámítása
        loss = criterion(outputs, batch_y)
        
        # Hibavisszaterjesztés (Backward pass - itt történik a tanulás)
        loss.backward()
        
        # Súlyok frissítése
        optimizer.step()
        
        running_loss += loss.item()
        
        # 500 batch-enként kiírjuk, hol tartunk, hogy ne legyen csendben
        if batch_idx % 500 == 0:
            print(f"Batch {batch_idx}/{len(train_loader)} | Aktuális Loss: {loss.item():.6f}")

    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    print(f"--- Epoch [{epoch+1}/{epochs}] kész. Átlagos hiba: {epoch_loss:.6f} | Idő: {duration:.2f}s ---")

Tanítás megkezdése a(z) cuda eszközön...
Batch 0/11555 | Aktuális Loss: 0.199582
Batch 500/11555 | Aktuális Loss: 0.005524
Batch 1000/11555 | Aktuális Loss: 0.007038
Batch 1500/11555 | Aktuális Loss: 0.006213
Batch 2000/11555 | Aktuális Loss: 0.005821
Batch 2500/11555 | Aktuális Loss: 0.005256
Batch 3000/11555 | Aktuális Loss: 0.005343
Batch 3500/11555 | Aktuális Loss: 0.005202
Batch 4000/11555 | Aktuális Loss: 0.005898
Batch 4500/11555 | Aktuális Loss: 0.004549
Batch 5000/11555 | Aktuális Loss: 0.005753
Batch 5500/11555 | Aktuális Loss: 0.005046
Batch 6000/11555 | Aktuális Loss: 0.003885
Batch 6500/11555 | Aktuális Loss: 0.004980
Batch 7000/11555 | Aktuális Loss: 0.007175
Batch 7500/11555 | Aktuális Loss: 0.004558
Batch 8000/11555 | Aktuális Loss: 0.005226
Batch 8500/11555 | Aktuális Loss: 0.006567
Batch 9000/11555 | Aktuális Loss: 0.005007
Batch 9500/11555 | Aktuális Loss: 0.004205
Batch 10000/11555 | Aktuális Loss: 0.005175
Batch 10500/11555 | Aktuális Loss: 0.004724
Batch 11000/115

In [5]:
# 1. Modell átállítása értékelési módba és adatok GPU-ra küldése
model_tcn.eval() 
all_preds = []
all_actuals = []

with torch.no_grad(): # Kikapcsoljuk a gradiens számítást (gyorsabb, kevesebb memória)
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        
        # Predikció
        outputs = model_tcn(batch_x)
        
        all_preds.append(outputs.cpu().numpy())
        all_actuals.append(batch_y.numpy())

# Tömbök összefűzése
y_pred = np.concatenate(all_preds, axis=0)
y_test = np.concatenate(all_actuals, axis=0)

# 2. Inverz skálázás (vissza Celsiusba)
# Mivel a scaler 10 oszlopra (features) vár adatot, de nekünk csak a predikciónk van:

def inverse_transform_multitarget(y_pred, y_test, scaler, target_idx, num_features):
    """
    y_pred: (N, horizon, num_targets)
    y_test: (N, horizon, num_targets)
    """
    num_targets = len(target_idx)
    N, H, _ = y_pred.shape

    y_pred_real = np.zeros_like(y_pred)
    y_test_real = np.zeros_like(y_test)

    for t in range(num_targets):
        idx = target_idx[t]

        # Laposítjuk a target t értékeit
        pred_flat = y_pred[:, :, t].reshape(-1)
        test_flat = y_test[:, :, t].reshape(-1)

        # Dummy mátrix a scaler számára
        dummy_pred = np.zeros((len(pred_flat), num_features))
        dummy_test = np.zeros((len(test_flat), num_features))

        dummy_pred[:, idx] = pred_flat
        dummy_test[:, idx] = test_flat

        # Inverz transzformáció
        inv_pred = scaler.inverse_transform(dummy_pred)[:, idx]
        inv_test = scaler.inverse_transform(dummy_test)[:, idx]

        # Visszaalakítás eredeti formára
        y_pred_real[:, :, t] = inv_pred.reshape(N, H)
        y_test_real[:, :, t] = inv_test.reshape(N, H)

    return y_pred_real, y_test_real


y_pred_real, y_test_real = inverse_transform_multitarget(y_pred, y_test, scaler, target_index, len(features))


In [6]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def calculate_metrics_overall(y_true, y_pred, model_name="Modell"):
    mse = mean_squared_error(y_true.reshape(-1), y_pred.reshape(-1))
    mae = mean_absolute_error(y_true.reshape(-1), y_pred.reshape(-1))
    rmse = np.sqrt(mse)

    print(f"--- {model_name} Összesített eredmények ---")
    print(f"MSE:  {mse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")

    return {"mse": mse, "mae": mae, "rmse": rmse}


def calculate_metrics_per_target(y_true, y_pred, target_names):
    num_targets = y_true.shape[2]
    results = {}

    print("\n--- Targetenkénti eredmények ---")
    for t in range(num_targets):
        mse = mean_squared_error(y_true[:,:,t].reshape(-1), y_pred[:,:,t].reshape(-1))
        mae = mean_absolute_error(y_true[:,:,t].reshape(-1), y_pred[:,:,t].reshape(-1))
        rmse = np.sqrt(mse)

        name = target_names[t]
        print(f"{name}: MAE={mae:.4f}, RMSE={rmse:.4f}")

        results[name] = {"mae": mae, "rmse": rmse}

    return results

def calculate_metrics_per_horizon(y_true, y_pred):
    N, H, T = y_true.shape
    mae_per_h = []
    rmse_per_h = []

    for h in range(H):
        true_h = y_true[:, h, :].reshape(-1)
        pred_h = y_pred[:, h, :].reshape(-1)

        mae = mean_absolute_error(true_h, pred_h)
        rmse = np.sqrt(mean_squared_error(true_h, pred_h))

        mae_per_h.append(mae)
        rmse_per_h.append(rmse)

    return np.array(mae_per_h), np.array(rmse_per_h)

def calculate_metrics_per_horizon_per_target(y_true, y_pred, target_names):
    N, H, T = y_true.shape
    results = {}

    for t in range(T):
        mae_list = []
        rmse_list = []

        for h in range(H):
            true_h = y_true[:, h, t]
            pred_h = y_pred[:, h, t]

            mae = mean_absolute_error(true_h, pred_h)
            rmse = np.sqrt(mean_squared_error(true_h, pred_h))

            mae_list.append(mae)
            rmse_list.append(rmse)

        results[target_names[t]] = {
            "mae": np.array(mae_list),
            "rmse": np.array(rmse_list)
        }

    return results



# TCN eredmények mentése
overall = calculate_metrics_overall(y_test_real, y_pred_real, "TCN")
per_target = calculate_metrics_per_target(y_test_real, y_pred_real, target_col)
mae_h, rmse_h = calculate_metrics_per_horizon(y_test_real, y_pred_real)
horizon_metrics = calculate_metrics_per_horizon_per_target(y_test_real, y_pred_real, target_col)


--- TCN Összesített eredmények ---
MSE:  15.4909
MAE:  2.2190
RMSE: 3.9359

--- Targetenkénti eredmények ---
temp_c: MAE=1.1423, RMSE=1.5645
rh: MAE=4.6341, RMSE=6.5265
wind_speed: MAE=0.8806, RMSE=1.1961


In [7]:
#TimesNET modell START

import torch.fft

class TimesNet(nn.Module):
    def __init__(self, input_window, num_features, output_horizon, num_targets, k=3):
        super(TimesNet, self).__init__()
        self.k = k
        self.input_window = input_window
        self.num_targets = num_targets
        self.output_horizon = output_horizon
        
        # Kezdő dimenzió növelés (hogy legyen elég "tere" a modellnek tanulni)
        self.embedding = nn.Linear(num_features, 64)
        
        # Inception-szerű 2D konvolúciós blokk
        self.conv = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1)
        )
        
        self.out_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * input_window, 128),
            nn.ReLU(),
            nn.Linear(128, output_horizon * num_targets)
        )

    def forward(self, x):
        # x: [Batch, Time, Features]
        B, T, D = x.size()
        x = self.embedding(x) # [B, T, 64]
        
        # 1. FFT a periódusok kereséséhez
        xf = torch.fft.rfft(x.transpose(1, 2), dim=-1)
        amplitudes = torch.mean(torch.abs(xf), dim=1)
        
        # Top-k periódus indexe
        _, top_k_indices = torch.topk(amplitudes[:, 1:], self.k, dim=-1)
        periods = T // (top_k_indices + 1)
        
        res = []
        for i in range(self.k):
            period = periods[0, i].item()
            # Padding
            p = (period - (T % period)) % period
            x_padded = torch.cat([x, torch.zeros([B, p, 64], device=x.device)], dim=1)
            
            # 2. Reshape 2D-re [B, D, H, W]
            H = (T + p) // period
            W = period
            x_2d = x_padded.reshape(B, H, W, 64).permute(0, 3, 1, 2)
            
            # 3. 2D konvolúció
            x_2d = self.conv(x_2d)
            
            # 4. Vissza 1D-be
            x_1d = x_2d.permute(0, 2, 3, 1).reshape(B, T + p, 64)
            res.append(x_1d[:, :T, :])
            
        # Összegzés és predikció
        x = torch.stack(res, dim=-1).mean(-1) + x
        
        out = self.out_layer(x)  # [B, output_horizon * num_targets]
        out = out.view(B, self.output_horizon, self.num_targets)
        return out


# Inicializálás GPU-ra
num_targets = len(target_index)

model_timesnet = TimesNet(
    input_window=input_window,
    num_features=len(features),
    output_horizon=output_horizon,
    num_targets=num_targets
).to(device)


In [ ]:
# 1. Beállítások a TimesNet-hez
criterion = nn.MSELoss()
optimizer = optim.Adam(model_timesnet.parameters(), lr=0.001)
epochs = 10 

print(f"TimesNet tanítása indul a(z) {device} eszközön...")

for epoch in range(epochs):
    start_time = time.time()
    model_timesnet.train() # Tanítási mód
    running_loss = 0.0
    
    for batch_idx, (batch_x, batch_y) in enumerate(train_loader):
        # Adatok GPU-ra
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        
        # Predikció
        outputs = model_timesnet(batch_x)
        
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # 500 batch-enként státuszjelentés
        if batch_idx % 500 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.6f}")

    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    print(f"--- TimesNet Epoch {epoch+1} Kész | Átlagos hiba: {epoch_loss:.6f} | Idő: {duration:.2f}s ---")

TimesNet tanítása indul a(z) cuda eszközön...
Epoch 1 | Batch 0/11555 | Loss: 0.194056
Epoch 1 | Batch 500/11555 | Loss: 0.008188
Epoch 1 | Batch 1000/11555 | Loss: 0.007033
Epoch 1 | Batch 1500/11555 | Loss: 0.006911
Epoch 1 | Batch 2000/11555 | Loss: 0.007196
Epoch 1 | Batch 2500/11555 | Loss: 0.005712
Epoch 1 | Batch 3000/11555 | Loss: 0.006261
Epoch 1 | Batch 3500/11555 | Loss: 0.005504
Epoch 1 | Batch 4000/11555 | Loss: 0.006145
Epoch 1 | Batch 4500/11555 | Loss: 0.005483


In [ ]:
# --- TIMESNET KIÉRTÉKELÉS ---

model_timesnet.eval()
all_preds_times = []
all_actuals_times = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model_timesnet(batch_x)

        all_preds_times.append(outputs.cpu().numpy())
        all_actuals_times.append(batch_y.numpy())

# Összefűzés
y_pred_times = np.concatenate(all_preds_times, axis=0)
y_test_times = np.concatenate(all_actuals_times, axis=0)

# --- HELYES több-targetes visszaskálázás ---
y_pred_times_real, y_test_times_real = inverse_transform_multitarget(
    y_pred_times, 
    y_test_times, 
    scaler, 
    target_index, 
    len(features)
)

print("TimesNet predikciók kész és visszaskálázva.")


In [ ]:
# Összesített hiba
timesnet_overall = calculate_metrics_overall(
    y_test_times_real, 
    y_pred_times_real, 
    model_name="TimesNet"
)

# Targetenkénti hiba
timesnet_per_target = calculate_metrics_per_target(
    y_test_times_real, 
    y_pred_times_real, 
    target_col
)

# Horizononkénti hiba (összes target átlagolva)
timesnet_mae_h, timesnet_rmse_h = calculate_metrics_per_horizon(
    y_test_times_real, 
    y_pred_times_real
)

# Horizon × Target mátrix
timesnet_horizon_metrics = calculate_metrics_per_horizon_per_target(
    y_test_times_real, 
    y_pred_times_real, 
    target_col
)


In [ ]:
import matplotlib.pyplot as plt

def plot_multitarget_predictions(y_true, y_pred, target_names, model_name, samples=500):
    """
    y_true, y_pred: (N, H, T)
    target_names: list of target names
    samples: hány időlépést rajzoljunk ki (flattenelve)
    """
    N, H, T = y_true.shape

    # Laposítjuk idő szerint, hogy folytonos görbét kapjunk
    true_flat = y_true.reshape(-1, T)
    pred_flat = y_pred.reshape(-1, T)

    # Csak az első 'samples' pontot rajzoljuk
    true_flat = true_flat[:samples]
    pred_flat = pred_flat[:samples]

    fig, axes = plt.subplots(T, 1, figsize=(15, 4*T), sharex=True)

    if T == 1:
        axes = [axes]  # ha csak 1 target van, legyen lista

    for t in range(T):
        axes[t].plot(true_flat[:, t], label="Valóság", color="black", linewidth=2, alpha=0.6)
        axes[t].plot(pred_flat[:, t], label=f"{model_name} jóslat", linestyle="--")
        axes[t].set_title(f"{model_name} – {target_names[t]}")
        axes[t].set_ylabel(target_names[t])
        axes[t].grid(True, alpha=0.3)
        axes[t].legend()

    axes[-1].set_xlabel("Időlépés (óra)")
    plt.tight_layout()
    plt.show()

plot_multitarget_predictions(
    y_test_real,
    y_pred_real,
    target_col,
    model_name="TCN"
)

plot_multitarget_predictions(
    y_test_times_real,
    y_pred_times_real,
    target_col,
    model_name="TimesNet"
)


In [ ]:
#Modellek mentése

# TCN mentése
torch.save(model_tcn.state_dict(), 'models4/tcn_model_weights.pth')

# TimesNet mentése
torch.save(model_timesnet.state_dict(), 'models4/timesnet_model_weights.pth')

print("Modellek súlyai elmentve!")

# # 1. Hozz létre egy ugyanolyan modellt
# loaded_model = TimesNet(input_window, len(features), output_horizon).to(device)

# # 2. Töltsd be a fájlt
# loaded_model.load_state_dict(torch.load('timesnet_model_weights.pth'))

# # 3. Állítsd eval módba (ha csak jósolni akarsz vele)
# loaded_model.eval()

# print("Modell sikeresen betöltve és kész a jóslásra!")

#Scaler mentése!
import joblib
joblib.dump(scaler, "models4/scaler.pkl")

#Visszatöltés:
#scaler = joblib.load("models/scaler.pkl")

#Features mentése
import json
json.dump(features, open("models4/features.json", "w"))
json.dump(target_index, open("models4/target_index.json", "w"))
json.dump({"num_targets": num_targets}, open("models4/model_params.json", "w"))

#Visszatöltés
#features = json.load(open("models/features.json"))
#target_index = json.load(open("models4/target_index.json"))
#params = json.load(open("models4/model_params.json"))
#num_targets = params["num_targets"]



#Windows paraméterek:
json.dump(
    {"input_window": input_window, "output_horizon": output_horizon},
    open("models4/window_params.json", "w")
)

#Visszatöltés
#params = json.load(open("models/window_params.json"))
#input_window = params["input_window"]
#output_horizon = params["output_horizon"]

